# Icechunk with annizarr

How to open, explore and version the AnnData store the `ingest` capsule writes. The store is an
[Icechunk](https://icechunk.io) repository read and written with
[annizarr](https://pypi.org/project/annizarr/), the same API `ingest/store.py` uses.

Run the cells top to bottom. Nothing here changes your store: reading is read-only, and section 7
writes a small demo repo to `/scratch` only.

## 1. Setup

Point `STORE` at any store:
- the ingest's output, `/results/ingest_icechunk` (after a run in this session);
- an attached data asset, such as `/data/sample_icechunk`;
- an S3 prefix, `s3://bucket/prefix`.

In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import annizarr
from annizarr import AppConfig, Repo, append, convert
from annizarr.config import IOConfig
from anndata.io import read_elem, sparse_dataset

print("annizarr", annizarr.__version__)

STORE = "/results/ingest_icechunk"
if not Repo.exists(STORE):
    STORE = "/data/sample_icechunk"  # falls back to the example data asset
print("Using", STORE)

## 2. Open the store

`Repo(path)` opens the repo; `open_zarr("r")` gives its root zarr group, which *is* the AnnData:
`X` (raw counts), `obs` (one row per cell) and `var` (one row per gene).

In [ ]:
repo = Repo(STORE)
root = repo.open_zarr("r")
n_cells, n_genes = root["X"].attrs["shape"]
print(f"{n_cells:,} cells x {n_genes:,} genes; groups: {sorted(root.keys())}")

## 3. History

Every ingest run is one commit (a *snapshot*). Its metadata names the samples it added.

In [ ]:
for snap in repo.log():
    print(snap.id, snap.written_at.strftime("%Y-%m-%d %H:%M"), "|", snap.message)

## 4. Cell metadata

`obs` is small enough to load whole, even for millions of cells. Use it to see what is in the store
and to pick cells.

In [ ]:
obs = read_elem(root["obs"])
print(obs.shape)
obs["sample_id"].value_counts().rename("cells").to_frame()

In [ ]:
# QC and doublets per sample, and the CellTypist labels if the store has them
per_sample = obs.groupby("sample_id", observed=True).agg(
    cells=("sample_id", "size"),
    mean_genes=("n_genes_by_counts", "mean"),
    mean_pct_mt=("pct_counts_mt", "mean"),
    doublets=("predicted_doublets", "sum"),
)
labels = [c for c in obs.columns if c.endswith("_label")]
print("CellTypist label columns:", labels or "none (the store was built without CellTypist)")
per_sample.round(1)

In [ ]:
if labels:
    label = labels[0]  # the coarsest model comes first, e.g. pbmc_clean_AIFI_L1_2024_label
    display(pd.crosstab(obs[label], obs["sample_id"]))

## 5. Load some cells as an AnnData

`X` is opened *backed*: only the rows you ask for are read. Pick rows with `obs`, then build an
AnnData for scanpy. Load what fits in memory, e.g. one sample at a time.

In [ ]:
X = sparse_dataset(root["X"])
var = read_elem(root["var"])

sample = obs["sample_id"].iloc[0]
rows = np.flatnonzero(obs["sample_id"].to_numpy() == sample)
cells = ad.AnnData(X=X[rows], obs=obs.iloc[rows].copy(), var=var)
cells

## 6. Time travel

Any snapshot in the log can be read as it was, e.g. to reproduce an analysis made before later
samples were added.

In [ ]:
first = repo.log()[-2]  # the oldest commit with data ([-1] is the empty "Repository initialized")
old = repo.open_zarr("r", snapshot_id=first.id)
print(f"{first.message!r}: {old['X'].attrs['shape'][0]:,} cells then, {n_cells:,} now")

## 7. Write your own repo (demo in /scratch)

What the ingest does under the hood: `convert` seeds a new repo with an AnnData as one commit;
`append(..., commit=False)` stages more cells and `repo.commit(...)` lands them as one snapshot
(`repo.discard()` would throw them away). Here a few hundred cells from above are written to a
throwaway repo.

In [ ]:
import shutil

DEMO = "/scratch/demo_icechunk_repo"
shutil.rmtree(DEMO, ignore_errors=True)

demo = Repo.create(DEMO)
seed_cfg = AppConfig(io=IOConfig(overwrite=True))  # lets convert fill the new, empty repo
convert(cells[:300], output=demo, cfg=seed_cfg, message="seed: 300 cells")

staged = "/scratch/demo_more_cells.zarr"
shutil.rmtree(staged, ignore_errors=True)
convert(cells[300:500], output=staged, cfg=AppConfig())
append(demo, cells=staged, cfg=AppConfig(), commit=False)
demo.commit("add 200 cells", metadata={"note": "notebook demo"})

print(demo.open_zarr("r")["X"].attrs["shape"])
for snap in demo.log():
    print(snap.id, "|", snap.message)

Branches work like git: `demo.checkout("test", create=True)` starts a branch where you can
append or edit without touching `main`; `demo.branches()` lists them and `demo.checkout("main")`
goes back.

In [ ]:
demo.checkout("test", create=True)
print(demo.branches())
demo.checkout("main")

## 8. Data assets linked to S3

A Code Ocean data asset is mounted read-only under `/data`. Reading works as above. To **write**
(that is what the ingest does when you fill in *Origin of s3*), name the bucket prefix the asset is
linked to:

```python
repo = Repo("/data/sample_linked_icechunk", origin="s3://bucket/prefix")
```

Commits then go to S3. The mount can lag S3 by a few commits, so to see the newest state read the
prefix directly: `Repo("s3://bucket/prefix").open_zarr("r")`.